# HandWash AI — entrenamiento robusto

Este notebook entrena un detector YOLO26s con el dataset público de 7 pasos. No reemplaza el modelo activo hasta que se copie manualmente el `best.pt` validado.

In [ ]:
!pip install -q ultralytics roboflow pyyaml
from pathlib import Path
PROJECT_DIR = Path('/content/PROYECTO-HAND-WASH-YOLO')
print('Sube o clona el proyecto en', PROJECT_DIR)

## Preparar el proyecto

Sube un ZIP del proyecto sin `node_modules`, `target` ni `DerivedData`, o reemplaza esta celda por el comando `git clone` de tu repositorio.

In [ ]:
from google.colab import files
import shutil, zipfile
if not PROJECT_DIR.exists():
    uploaded = files.upload()
    zip_name = next(name for name in uploaded if name.endswith('.zip'))
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name) as archive:
        archive.extractall('/content/unpacked')
    candidates = list(Path('/content/unpacked').glob('**/scripts/train_handwash_robust.py'))
    if not candidates:
        raise FileNotFoundError('El ZIP no contiene scripts/train_handwash_robust.py')
    extracted_root = candidates[0].parents[1]
    shutil.copytree(extracted_root, PROJECT_DIR, dirs_exist_ok=True)
print((PROJECT_DIR / 'scripts/train_handwash_robust.py').exists())

In [ ]:
import os, getpass
try:
    from google.colab import userdata
    os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    os.environ['ROBOFLOW_API_KEY'] = getpass.getpass('ROBOFLOW_API_KEY (no se mostrará): ')
assert os.environ['ROBOFLOW_API_KEY'], 'Falta ROBOFLOW_API_KEY'

In [ ]:
%cd /content/PROYECTO-HAND-WASH-YOLO
!python scripts/train_handwash_robust.py --epochs 120 --imgsz 640 --batch 16 --device 0 --install-model

In [ ]:
from google.colab import files
candidate = PROJECT_DIR / 'backend/models/handwash_yolo26s_robust.pt'
manifest = PROJECT_DIR / 'backend/models/model-manifest.json'
assert candidate.exists(), candidate
assert manifest.exists(), manifest
files.download(str(candidate))
files.download(str(manifest))
print('Lleva ambos archivos al proyecto. El .pt es un detector candidato; no reemplaza ni activa el modelo actual.')